## Limitations
- The dataset is relatively small (2,000 rows before the cleaning and 1996 rows after the cleaning), limiting the model's ability 
  to generalize across diverse student populations.
- GPA is influenced by many factors not present in this dataset 
  (e.g. intelligence, teaching quality, motivation).
- Model R² of ~0.53 indicates that approximately 47% of GPA variance 
  remains unexplained by the selected features.
- Cross-validation std of ±0.025 reflects sensitivity to data composition 
  due to the small sample size.
- Predictions should be used as a rough estimate only.

In [1]:
import pandas as pd

df = pd.read_csv('datasets/student_lifestyle_dataset.csv')

In [2]:
df = df.drop(columns=['Student_ID'])

df = df.rename(columns={
    "Study_Hours_Per_Day": "study_hours",
    "Extracurricular_Hours_Per_Day": "eca_hours",
    "Sleep_Hours_Per_Day": "sleep_hours",
    "Social_Hours_Per_Day": "social_hours",
    "Physical_Activity_Hours_Per_Day": "physical_hours",
    "Stress_Level": "stress_level",
    "GPA": "gpa"
})

In [3]:
from scipy import stats
import numpy as np

df = df[(np.abs(stats.zscore(df.select_dtypes("number"))) < 3).all(axis=1)] 
print(f'After cleaning: {len(df)}')

After cleaning: 1996


In [4]:
df["stress_level"].unique()

array(['Moderate', 'Low', 'High'], dtype=object)

In [5]:
import joblib

df["stress_level"] = df["stress_level"].map({"Low": 0, "Moderate": 1, "High": 2})

stress_mapping = {"Low": 0, "Moderate": 1, "High": 2}
joblib.dump(stress_mapping, "models/stress_level_encoder_modelB.pkl")

['models/stress_level_encoder_modelB.pkl']

In [6]:
x = df.drop("gpa", axis=1)
y = df["gpa"]

In [7]:
from sklearn.model_selection import train_test_split


x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

In [8]:
from sklearn.ensemble import GradientBoostingRegressor

model_b = GradientBoostingRegressor(
    n_estimators=1000,
    max_depth=2,
    learning_rate=0.01,
    subsample=0.9,  
    max_features=0.7,  
    n_iter_no_change=50,  
    validation_fraction=0.1,  
    min_samples_leaf=3,
    tol=1e-4,
    random_state=42,
)

model_b.fit(x_train, y_train)
y_pred = model_b.predict(x_test)

In [9]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE  : {mse}")
print(f"RMSE : {rmse}")
print(f"MAE  : {mae}")
print(f"R2   : {r2}")

joblib.dump(model_b, "models/modelB.pkl")
print("Model B is successfully saved!")

actual_range_b = y_test.max() - y_test.min()
nrmse_b = rmse / actual_range_b
print(f"y_test range: {actual_range_b:.4f}")
print(f"NRMSE (actual range): {nrmse_b:.4f} = {nrmse_b*100:.2f}%")

MSE  : 0.04118571582193026
RMSE : 0.20294264170432555
MAE  : 0.16464878350039933
R2   : 0.5333078064908834
Model B is successfully saved!
y_test range: 1.7000
NRMSE (actual range): 0.1194 = 11.94%


In [10]:
from sklearn.model_selection import cross_val_score, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_r2   = cross_val_score(model_b, x, y, cv=kf, scoring="r2")
cv_mae  = cross_val_score(model_b, x, y, cv=kf, scoring="neg_mean_absolute_error")
cv_rmse = cross_val_score(model_b, x, y, cv=kf, scoring="neg_root_mean_squared_error")

print(f"CV R²   : {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"CV MAE  : {(-cv_mae).mean():.4f} ± {(-cv_mae).std():.4f}")
print(f"CV RMSE : {(-cv_rmse).mean():.4f} ± {(-cv_rmse).std():.4f}")

CV R²   : 0.5289 ± 0.0246
CV MAE  : 0.1622 ± 0.0038
CV RMSE : 0.2042 ± 0.0041
